# Orchestrateur agentique V9 — modèle unique 14B (GPU Colab)

Même architecture multi-agents que `_bench_orchestrator.py` (bench **V9 hybride**), mais **un seul modèle 14B pour TOUTES les phases**. Sur GPU au lieu du CPU local.

## Runtime
`Exécution > Modifier le type d'exécution > GPU > T4` (gratuit, suffisant). L4/A100 = plus de marge.

## À uploader (cellule 5) — SEULEMENT 3 fichiers à plat, AUCUN dossier
1. `meeting_minutes_pipeline.py`
2. `_bench_orchestrator.py`
3. ton transcript `.txt` (ex. `dicte_audio_3.normalized.txt`)

`colab_run.py` est écrit automatiquement par la cellule 6 — rien à uploader pour lui.
Exécute les cellules dans l'ordre.

## 1. Vérifier le GPU

In [ ]:
!nvidia-smi

## 2. Dépendances (numpy/scipy/torch déjà présents sur Colab)

In [ ]:
!pip install -q sentence-transformers psutil huggingface_hub
import torch; print('CUDA :', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 3. Compiler llama.cpp avec CUDA (~6-10 min)
On ne compile que la cible `llama-server`. `-DLLAMA_CURL=OFF` évite la dépendance libcurl.

In [ ]:
%cd /content
!nvcc --version | tail -n 2
![ -d llama.cpp ] || git clone --depth 1 https://github.com/ggml-org/llama.cpp
!cmake -S llama.cpp -B llama.cpp/build -DGGML_CUDA=ON -DLLAMA_CURL=OFF
!cmake --build llama.cpp/build --config Release --target llama-server -j $(nproc)
!ls -lh /content/llama.cpp/build/bin/llama-server

## 4. Télécharger le modèle 14B (GGUF Q4_K_M ~9 Go)
Défaut **Qwen2.5-14B-Instruct** (bon en français + sortie JSON fiable). Alternatives dans le README.

In [ ]:
from huggingface_hub import hf_hub_download
import os
os.makedirs('/content/models', exist_ok=True)
MODEL = hf_hub_download(
    repo_id='bartowski/Qwen2.5-14B-Instruct-GGUF',
    filename='Qwen2.5-14B-Instruct-Q4_K_M.gguf',
    local_dir='/content/models')
print('Modele :', MODEL)

## 5. Uploader les 3 fichiers (à plat)
Lance la cellule, puis sélectionne `meeting_minutes_pipeline.py`, `_bench_orchestrator.py` et ton transcript.

In [ ]:
%cd /content
from google.colab import files
up = files.upload()
print('Uploades :', list(up))

## 6. Écrire le runner `colab_run.py` (automatique, rien à uploader)

In [ ]:
%%writefile colab_run.py
# Runner auto-genere par le notebook. Lance llama-server en mode GPU (CUDA,
# -ngl 99) puis appelle l'orchestrateur agentique en MODE MODELE UNIQUE :
# seul --model est fourni => routing_actif=False => le 14B fait TOUTES les
# phases (extraction, Context Builder, Planner, Designers, Workers/Juges).
# Aucune modification des fichiers source : on monkeypatch start_llm_server_slots.
import argparse, atexit, subprocess, sys, time, urllib.request
from pathlib import Path
import meeting_minutes_pipeline as mmp
import _bench_orchestrator as bench


def make_launcher(server_bin, ngl, ctx):
    def start(cfg, parallel_slots):
        try:
            subprocess.run(["fuser", "-k", f"{cfg.llm_server_port}/tcp"], capture_output=True)
        except Exception:
            pass
        if not Path(server_bin).exists():
            raise FileNotFoundError(f"llama-server introuvable : {server_bin}")
        if not Path(cfg.llm_model_path).exists():
            raise FileNotFoundError(f"Modele introuvable : {cfg.llm_model_path}")
        total_ctx = ctx if ctx > 0 else (cfg.llm_n_ctx or 16384)
        cmd = [server_bin, "-m", cfg.llm_model_path,
               "--port", str(cfg.llm_server_port),
               "--ctx-size", str(total_ctx),
               "--parallel", str(parallel_slots),
               "-ngl", str(ngl),
               "--flash-attn", "on",
               "--cache-ram", "0",
               "--cache-type-k", cfg.llm_kv_cache_type,
               "--cache-type-v", cfg.llm_kv_cache_type,
               "--batch-size", "4096",
               "--ubatch-size", "1024",
               "--log-disable"]
        print("[GPU] llama-server :", " ".join(cmd), flush=True)
        fh = open("_llama_server_stderr.log", "w", encoding="utf-8", errors="replace")
        proc = subprocess.Popen(cmd, stdout=subprocess.DEVNULL, stderr=fh)
        mmp._server_process = proc
        atexit.register(mmp._kill_server)
        health = f"http://{cfg.llm_server_host}:{cfg.llm_server_port}/health"
        deadline = time.time() + 600
        while time.time() < deadline:
            try:
                with urllib.request.urlopen(health, timeout=2) as r:
                    if r.status == 200:
                        print("[GPU] llama-server pret", flush=True)
                        return
            except Exception:
                pass
            if proc.poll() is not None:
                fh.flush()
                tail = Path("_llama_server_stderr.log").read_text(encoding="utf-8", errors="replace")[-3000:]
                raise RuntimeError("llama-server a crashe au demarrage. Log : " + tail)
            time.sleep(1)
        raise TimeoutError("llama-server timeout au demarrage (600s)")
    return start


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--server-bin", required=True)
    ap.add_argument("--model", required=True)
    ap.add_argument("--transcript", required=True)
    ap.add_argument("--participants", default=bench.DEFAULT_PARTICIPANTS)
    ap.add_argument("--entreprises", default="")
    ap.add_argument("--output-dir", default="out_14b")
    ap.add_argument("--ngl", type=int, default=99)
    ap.add_argument("--ctx", type=int, default=16384)
    a = ap.parse_args()
    launcher = make_launcher(a.server_bin, a.ngl, a.ctx)
    bench.start_llm_server_slots = launcher
    mmp.start_llm_server_slots = launcher
    print("[INFO] Modele unique (toutes phases) :", Path(a.model).name, "| -ngl", a.ngl, "| ctx", a.ctx)
    return bench.run(transcript_path=Path(a.transcript), sections_path=None,
                     participants=a.participants, entreprises=a.entreprises,
                     output_dir=Path(a.output_dir), agentic_model=Path(a.model),
                     context_model=None, worker_model=None, draft_model=None)


if __name__ == "__main__":
    sys.exit(main())


## 7. Lancer le pipeline complet
**Adapte** `--participants` (noms EXACTS) et `--transcript` (nom de ton fichier). Baisse `--ctx 8192` si OOM VRAM sur T4.

In [ ]:
!cd /content && python colab_run.py \
  --server-bin /content/llama.cpp/build/bin/llama-server \
  --model /content/models/Qwen2.5-14B-Instruct-Q4_K_M.gguf \
  --transcript /content/dicte_audio_3.normalized.txt \
  --participants "Bruno LEMETAYER, Matthieu DUSSARTRE, Nourredine HENKA, Jerome PICAULT, Maya SAHRAOUI, Jerome MASSET" \
  --entreprises "" \
  --output-dir /content/out_14b \
  --ngl 99 --ctx 16384

## 8. Afficher le compte rendu

In [ ]:
from pathlib import Path
from IPython.display import Markdown, display
display(Markdown(Path('/content/out_14b/compte_rendu_v4.md').read_text(encoding='utf-8')))

## 9. Télécharger les résultats

In [ ]:
from google.colab import files
files.download('/content/out_14b/compte_rendu_v4.md')
files.download('/content/out_14b/orchestrator_v4.json')